In [1]:
from llama_index.core import VectorStoreIndex, ServiceContext
from llama_index.llms.gemini import Gemini

c:\Users\Toby\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Toby\miniconda3\envs\rag\Lib\site-packages\llama_index\llms\gemini\base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import fitz  # pymupdf

pdf = fitz.open("eBook-How-to-Build-a-Career-in-AI.pdf")
text = "\n\n".join([page.get_text() for page in pdf])

print(text[:1000])

PAGE 1
Founder, DeepLearning.AI
Collected Insights
from Andrew Ng
How to 
Build
Your
Career
in AI
A Simple Guide


PAGE 2
"AI is the new 
electricity. It will 
transform and improve 
all areas of human life."
Andrew Ng


PAGE 3
Table of 
Contents
Introduction: Coding AI is the New Literacy.
Chapter 1: Three Steps to Career Growth.
Chapter 2: Learning Technical Skills for a 
Promising AI Career.
Chapter 3: Should You Learn Math to Get a Job 
in AI?
Chapter 4: Scoping Successful AI Projects.
Chapter 5: Finding Projects that Complement 
Your Career Goals.
Chapter 6: Building a Portfolio of Projects that 
Shows Skill Progression.
Chapter 7: A Simple Framework for Starting Your AI 
Job Search.
Chapter 8: Using Informational Interviews to Find 
the Right Job.
Chapter 9: Finding the Right AI Job for You.
Chapter 10: Keys to Building a Career in AI.
Chapter 11: Overcoming Imposter Syndrome.
Final Thoughts: Make Every Day Count.
LEARNING
PROJECTS
JOB


PAGE 4
Coding AI Is the New Literacy
Today

In [3]:
from llama_index.core import Document

document = Document(text=text)

In [4]:
import utils

from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.gemini import Gemini

Settings.llm = Gemini(
    model="models/gemini-2.5-flash",
    api_key=utils.get_gemini_api_key(),
    temperature=0.1
)
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

index = VectorStoreIndex.from_documents([document], show_progress=True)


C:\Users\Toby\AppData\Local\Temp\ipykernel_23544\1190761636.py:6: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(
Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  6.64it/s]


In [5]:
index.storage_context.persist(persist_dir="./storage")


In [6]:
import os
from llama_index.core import StorageContext, load_index_from_storage, VectorStoreIndex

if os.path.exists("./storage"):
    storage_context = StorageContext.from_defaults(persist_dir="./storage")
    index = load_index_from_storage(storage_context)
else:
    index = VectorStoreIndex.from_documents([document], show_progress=True)
    index.storage_context.persist(persist_dir="./storage")

In [7]:
query_engine = index.as_query_engine(similarity_top_k=5)

# increasing # of chunks received leads to more context for the LLM to reason over, meaning lower risk of missing relevant info, but higher risk of irrelevant info being included

# decreasing # of chunks leads to only most confident matches going through, tighter more focused answer but higher risk of missing relevant info

In [8]:
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

To find and scope projects that build experience, consider the following steps and strategies:

First, for scoping successful projects:
*   **Identify a business problem:** Focus on real-world business challenges rather than immediately thinking of AI solutions. Engage with domain experts to understand their top pain points and why current solutions are insufficient.
*   **Brainstorm AI solutions:** Once a problem is clearly defined, explore various potential AI-driven approaches. It's important to consider multiple options before settling on one, and recognize that sometimes an AI solution may not be the best fit.
*   **Determine milestones:** Establish clear metrics for success, encompassing both machine learning performance indicators (such as accuracy) and relevant business metrics (like revenue or user engagement). If milestones are hard to define, it may indicate a need for deeper understanding of the problem, which a quick proof of concept can help provide.
*   **Assess feasibil

<div style="color:#2563eb">

### Chunking

Testing different chunk size

</div>

In [10]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)

index = VectorStoreIndex.from_documents([document], show_progress=True)

Generating embeddings: 100%|██████████| 12/12 [00:02<00:00,  5.19it/s]


In [11]:
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

To find projects and build experience, a structured approach to scoping can be followed, along with methods for generating project ideas.

For scoping projects:
1.  **Identify a business problem:** Focus on real-world issues rather than AI-specific challenges. Engage with domain experts to understand their top pain points and the reasons behind them.
2.  **Brainstorm AI solutions:** Once the problem is clear, explore various potential AI-driven solutions. It's important to consider multiple options before settling on one.
3.  **Determine milestones:** Establish clear metrics for success, encompassing both machine learning metrics (e.g., accuracy) and business metrics (e.g., revenue, user engagement). If milestones are hard to define, more understanding of the problem may be needed, possibly through a proof of concept.
4.  **Assess feasibility and value:** Evaluate if a proposed solution is technically achievable by reviewing existing work or competitor actions, or by building a quick p

<div style="color:#2563eb">

Differences:

Longer responses in response 2, could be due to larger chunk overlap (20 in response 1 vs 50 in response 2)
Ideas presented in the responses are still the same, just presented a little differently

Response 2 separates the response into 2 lists rather than 1 long sentence + 1 list

</div>

<div style="color:#2563eb">

### Semantic chunking

Separate chunks when the meaning of the chunks start to change, not just by chunk size

</div>

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core import Settings

Settings.node_parser = SemanticSplitterNodeParser(
    
    buffer_size=1,              # how many sentences to group before comparing, ex: sentence 1 vs sentence 2

    breakpoint_percentile_threshold=95,  # higher = fewer, larger chunks, top 5% of breakpoints are used
    
    embed_model=Settings.embed_model    # reuses your existing bge-small embedder
)

In [13]:
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

To find projects that help build experience, consider the following approaches:

One effective method is to identify a business problem rather than an AI problem. This can be done by consulting a domain expert and inquiring about the top three areas they wish worked better and why they currently do not.

Additionally, if you are looking for project ideas, you can:
*   Join existing projects that others have initiated.
*   Continuously read and engage in conversations with people, especially domain experts, to generate new ideas.
*   Concentrate on a specific application area, particularly those where machine learning has not yet been extensively applied.
*   Develop a side hustle, which can be a passion project that may evolve into something more significant.


<div style="color:#2563eb">

Differences:

Shorter response, more similar in length to response 1, but is the most concise.

This is due to the high breakpoint threshold, end up with fewer larger chunks

Semantic chunking creates breakpoints only at bigger topic shifts, previous responses that included the "5 step process" chunk might have gotten kicked out of the retrieval window because it's not different enough

</div>

<div style="color:#2563eb">

### Testing different embedding models:

All-mpnet-base-v2

</div>

In [14]:
from llama_index.core import Settings

Settings.embed_model = "local:sentence-transformers/all-mpnet-base-v2"

c:\Users\Toby\miniconda3\envs\rag\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Toby\AppData\Local\llama_index\llama_index\Cache\models\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6627.07it/s]

In [15]:
index = VectorStoreIndex.from_documents([document], show_progress=True)


Generating embeddings: 100%|██████████| 12/12 [00:04<00:00,  2.80it/s]


In [16]:

Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)

In [17]:
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

When seeking projects to build experience, several approaches can help generate ideas and then refine them into actionable plans.

To generate project ideas:
*   Join existing projects if you find someone else with an idea.
*   Continuously read and engage in conversations with domain experts.
*   Concentrate on a specific application area where machine learning may not yet be widely applied.
*   Consider developing a side hustle, which can be a fun project that might evolve into something more significant.

Once you have a few project ideas, follow these steps to scope them:
1.  Identify a business problem, rather than an AI problem, by consulting domain experts about what they wish worked better.
2.  Brainstorm various AI solutions for the identified problem.
3.  Assess the feasibility and value of these potential solutions, which can involve reviewing published work, observing competitors, or creating a quick proof of concept.
4.  Determine clear milestones, encompassing both machin

<div style="color:#2563eb">

Matched original fixed size chunking result from the BAAI embedding model

Confirms that the semantic chunking strategy changed the response since that response doesn't include the 5-step process

</div>

<div style="color:#2563eb">

### Semantic chunking with mpnet embedding model

</div>

In [18]:
from llama_index.core import Settings
from llama_index.core.node_parser import SemanticSplitterNodeParser

Settings.embed_model = "local:sentence-transformers/all-mpnet-base-v2"

Settings.node_parser = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model  # semantic chunking will now use mpnet too
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7362.84it/s]


In [19]:
index = VectorStoreIndex.from_documents([document], show_progress=True)


Generating embeddings: 100%|██████████| 12/12 [00:04<00:00,  2.81it/s]


In [20]:
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

To find projects that build experience and align with career goals, several approaches can be taken to generate ideas and then define them:

To generate project ideas:
*   Join existing projects if someone else has an idea.
*   Continuously read and engage in conversations with domain experts to spark new ideas.
*   Focus on an application area where machine learning has not yet been widely applied, exploring possibilities within a company or school's interests.
*   Develop a side hustle, which can be a fun project pursued out of passion, potentially leading to something significant.

Once general ideas are considered, a structured approach to scoping projects involves:
*   Identifying a business problem by consulting domain experts about areas needing improvement.
*   Brainstorming various AI solutions for the identified problem.
*   Evaluating the technical feasibility and potential value of these solutions, which might involve reviewing published work, competitor actions, or buildin

<div style="color:#2563eb">

mpnet with semantic chunking gives a similar response to fixed chunking

different result than with the BAAI embedding model that had different responses for the 2 chunking methods

mpnet uses 768 dimension embeddings to discriminate, that could be the root cause

</div>

<div style="color:#2563eb">

### Performance evaluation

</div>

In [22]:
for i, node in enumerate(response.source_nodes):
    print(f"--- Chunk {i+1} (score: {node.score:.3f}) ---")
    print(node.text[:300])
    print()


--- Chunk 1 (score: 0.580) ---
PAGE 17
Finding Projects that 
Complement Your 
Career Goals
CHAPTER 5
PROJECTS


PAGE 18
It goes without saying that we should only work on projects that are responsible, ethical, and 
beneficial to people. But those limits leave a large variety to choose from. In the previous chapter, 
I wrote abo

--- Chunk 2 (score: 0.542) ---
PAGE 23
Each project is only one step on a longer journey, hopefully one that has a positive impact. In addition:
Don’t worry about starting too small. One of my first machine learning research projects involved 
training a neural network to see how well it could mimic the sin(x) function. It wasn’t

--- Chunk 3 (score: 0.524) ---
PAGE 14
Scoping Successful 
AI Projects
CHAPTER 4
PROJECTS


PAGE 15
One of the most important skills of an AI architect is the ability to identify ideas that are worth 
working on. These next few chapters will discuss finding and working on projects so you can gain 
experience and build your portfo

-

<div style="color:#2563eb">

compare side by side with BAAI embedding model chunks

</div>

In [23]:
from llama_index.core import Settings
from llama_index.core.node_parser import SemanticSplitterNodeParser

Settings.embed_model = "local:BAAI/bge-small-en-v1.5"
Settings.node_parser = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model
)

index = VectorStoreIndex.from_documents([document], show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)

for i, node in enumerate(response.source_nodes):
    print(f"--- Chunk {i+1} (score: {node.score:.3f}) ---")
    print(node.text[:300])
    print()

Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  8.67it/s]


--- Chunk 1 (score: 0.750) ---
PAGE 14
Scoping Successful 
AI Projects
CHAPTER 4
PROJECTS


PAGE 15
One of the most important skills of an AI architect is the ability to identify ideas that are worth 
working on. These next few chapters will discuss finding and working on projects so you can gain 
experience and build your portfo

--- Chunk 2 (score: 0.749) ---
PAGE 17
Finding Projects that 
Complement Your 
Career Goals
CHAPTER 5
PROJECTS


PAGE 18
It goes without saying that we should only work on projects that are responsible, ethical, and 
beneficial to people. But those limits leave a large variety to choose from. In the previous chapter, 
I wrote abo

--- Chunk 3 (score: 0.741) ---
PAGE 23
Each project is only one step on a longer journey, hopefully one that has a positive impact. In addition:
Don’t worry about starting too small. One of my first machine learning research projects involved 
training a neural network to see how well it could mimic the sin(x) function. It wasn’t

-

<div style="color:#2563eb">

### Differences between the BAAI chunks and mpnet chunks:

Ranking order:

- BAAI ranks page 14, 17, then 23
- mpnet ranks 17, 23, then 14

Score ranges:

- BAAI score ranges closer to 0.7
- mpnet around 0.4 - 0.5

^ these results don't mean baai is more accurate or confident, the confidence scores matter within each model's own results

</div>

<div style="color:#2563eb">

Also since page 14 content (that has the "5 step process") was ranked #1 in the BAAI model then the difference between the semantic vs chunking outputs were probably just llm output generation variance

</div>

<div style="color:#2563eb">

### test whether RAG pipeline retrieves the right source chunk for a set of questions

</div>

In [26]:
import time

# Define your test set: question + a marker string that should appear in the correct chunk
test_questions = [
    {
        "question": "What are steps to take when finding projects to build your experience?",
        "correct_marker": "PAGE 14"  # adjust if you confirm a different page is the best match
    },
    {
        "question": "What should I consider when choosing between two different project directions?",
        "correct_marker": "PAGE 20"
    },
    {
        "question": "What tips help with finding the right AI job?",
        "correct_marker": "PAGE 31"
    },
    {
        "question": "What are the three steps to career growth in AI?",
        "correct_marker": "PAGE 7"
    },
]

def reciprocal_rank(source_nodes, correct_marker):
    for i, node in enumerate(source_nodes):
        if correct_marker in node.text:
            return 1 / (i + 1)
    return 0

def run_eval(query_engine, test_questions, config_name):
    results = []
    for item in test_questions:
        start = time.time()
        response = query_engine.query(item["question"])
        elapsed = time.time() - start

        rr = reciprocal_rank(response.source_nodes, item["correct_marker"])
        hit = 1 if rr > 0 else 0

        results.append({
            "config": config_name,
            "question": item["question"][:50] + "...",
            "hit": hit,
            "reciprocal_rank": round(rr, 3),
            "time_sec": round(elapsed, 2)
        })
    return results

# Run it
query_engine = index.as_query_engine(similarity_top_k=5)
results = run_eval(query_engine, test_questions, config_name="bge-small + semantic")

# Print as a simple table
for r in results:
    print(r)

# Summary stats
avg_hit_rate = sum(r["hit"] for r in results) / len(results)
avg_mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
print(f"\nHit Rate: {avg_hit_rate:.2f}")
print(f"MRR: {avg_mrr:.2f}")

{'config': 'bge-small + semantic', 'question': 'What are steps to take when finding projects to bu...', 'hit': 1, 'reciprocal_rank': 1.0, 'time_sec': 4.99}
{'config': 'bge-small + semantic', 'question': 'What should I consider when choosing between two d...', 'hit': 1, 'reciprocal_rank': 1.0, 'time_sec': 4.83}
{'config': 'bge-small + semantic', 'question': 'What tips help with finding the right AI job?...', 'hit': 1, 'reciprocal_rank': 1.0, 'time_sec': 7.03}
{'config': 'bge-small + semantic', 'question': 'What are the three steps to career growth in AI?...', 'hit': 1, 'reciprocal_rank': 0.333, 'time_sec': 1.53}

Hit Rate: 1.00
MRR: 0.83


<div style="color:#2563eb">

Hit rate: did it receive the right source chunk

MRR: how high did it rank when the source chunk was found?

The rank for the source chunk in the last question was not ranked #1 which is why it got a lower RR score

</div>